# Cross-cell-line scenario construction

For each folder named `Multi_5_to_<cell line>`, the matching benchmark file under `data/SLbench/Source/<cell line>/` is used as the **test** set, while all remaining cell-line benchmark files are concatenated into the **training** set.

The cell-line id stored in the third column follows the order defined in `data/SLbench/Source/cell_line.txt`:

- `A375 -> 0`
- `A549 -> 1`
- `Jurkat -> 2`
- `MeWo -> 3`
- `22Rv1 -> 4`
- `Pk1 -> 5`

The path configuration cell searches upward from the current working directory, so the notebook can be run from the repository root or from `tutorials/`.


In [ ]:
from pathlib import Path

import pandas as pd


def find_existing_path(*parts: str) -> Path:
    """Search upward from the current working directory until the requested path exists."""
    for root in (Path.cwd(), *Path.cwd().parents):
        candidate = root.joinpath(*parts)
        if candidate.exists():
            return candidate.resolve()
    raise FileNotFoundError(f'Could not locate path: {Path(*parts)}')


SOURCE_ROOT = find_existing_path('data', 'SLbench', 'Source')
SCENARIO_ROOT = find_existing_path('data', 'SLbench', 'Scenario', 'Cross_cell_line')
CELL_LINE_FILE = SOURCE_ROOT / 'cell_line.txt'


In [ ]:
# Read the canonical cell-line order and build the id mapping.
CELL_LINES = [line.strip() for line in CELL_LINE_FILE.read_text().splitlines() if line.strip()]
CELL_LINE_TO_ID = {cell_line: idx for idx, cell_line in enumerate(CELL_LINES)}

CELL_LINE_TO_ID


In [ ]:
def find_bench_file(cell_line: str) -> Path:
    """Return the unique benchmark file for one cell line."""
    bench_files = sorted((SOURCE_ROOT / cell_line).glob('*_bench.csv'))
    if len(bench_files) != 1:
        raise FileNotFoundError(
            f'Expected exactly one *_bench.csv file for {cell_line}, found {len(bench_files)}.'
        )
    return bench_files[0]


def load_bench(cell_line: str) -> pd.DataFrame:
    """Load one cell-line benchmark file and enforce the id from cell_line.txt order."""
    bench_path = find_bench_file(cell_line)
    df = pd.read_csv(bench_path).copy()
    df.iloc[:, 2] = CELL_LINE_TO_ID[cell_line]
    return df


def get_output_dir(target_cell_line: str) -> Path:
    """Return the scenario folder for one target cell line."""
    return SCENARIO_ROOT / f'Multi_5_to_{target_cell_line}'


def build_split_for_target(target_cell_line: str) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Build one cross-cell-line split.

    Example:
        For `Multi_5_to_A375`, `A375/a375_bench.csv` is used as the test set.
        All other benchmark files are concatenated into the training set and
        the training index is reset before saving.
    """
    test_df = load_bench(target_cell_line).reset_index(drop=True)
    train_parts = [
        load_bench(cell_line)
        for cell_line in CELL_LINES
        if cell_line != target_cell_line
    ]
    train_df = pd.concat(train_parts, ignore_index=True).reset_index(drop=True)

    output_dir = get_output_dir(target_cell_line)
    output_dir.mkdir(parents=True, exist_ok=True)
    train_df.to_csv(output_dir / 'sl_train_0.csv', index=False)
    test_df.to_csv(output_dir / 'sl_test_0.csv', index=False)

    return train_df, test_df


def build_all_cross_cell_line_scenarios() -> pd.DataFrame:
    """Rebuild every cross-cell-line scenario folder and return a summary table."""
    summary_rows = []
    for target_cell_line in CELL_LINES:
        train_df, test_df = build_split_for_target(target_cell_line)
        summary_rows.append(
            {
                'target_cell_line': target_cell_line,
                'target_id': CELL_LINE_TO_ID[target_cell_line],
                'train_rows': len(train_df),
                'test_rows': len(test_df),
                'output_dir': str(get_output_dir(target_cell_line)),
            }
        )
    return pd.DataFrame(summary_rows)


In [ ]:
# Run this cell to rebuild all `sl_train_0.csv` and `sl_test_0.csv` files.
build_all_cross_cell_line_scenarios()
